# E1-03 · Laboratorio 2 · Teorema de Bayes con datos de accidentes de tránsito

**Estadística I · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión estudiante*

**Propósito de aprendizaje**

- Definir eventos probabilísticos a partir de las columnas de una base de datos real.
- Calcular probabilidades simples, conjuntas y condicionales con frecuencias relativas.
- Aplicar el teorema de Bayes, verificarlo con el cálculo directo e interpretar cómo cambia una probabilidad al incorporar evidencia.

**Duración sugerida:** 1 sesión de 2 horas + trabajo autónomo  
**Datos:** Accidentes de tránsito en Bucaramanga, Datos Abiertos Colombia (`7cci-nqqb`)

## Caso de estudio · Factores asociados a la gravedad de los accidentes

La base **“Accidentes de Tránsito ocurridos en el Municipio de Bucaramanga”** ([Datos Abiertos Colombia](https://www.datos.gov.co/Transporte/3-Accidentes-de-Transito-ocurridos-en-el-Municipio/7cci-nqqb)) contiene los siniestros viales registrados en la ciudad: la gravedad del accidente, el momento del día, los actores viales involucrados y la ubicación (barrio y comuna).

La Dirección de Tránsito de Bucaramanga quiere fortalecer sus estrategias de prevención identificando si el horario, la participación de ciertos actores viales y la ubicación están relacionados con una mayor gravedad. Para ello solicita responder:

1. ¿Cuál es la probabilidad de que un accidente tenga heridos?
2. ¿Cuál es la probabilidad de que un accidente ocurra en horario nocturno?
3. ¿Cuál es la probabilidad de que un accidente tenga heridos, **dado que** ocurrió en horario nocturno?
4. ¿Cuál es la probabilidad de que un accidente haya ocurrido en horario nocturno, **dado que** tuvo heridos?
5. ¿Cuál es la probabilidad de que un accidente tenga heridos, dado que hubo participación de motocicleta?
6. ¿Cuál es la probabilidad de que un accidente tenga muertos, dado que hubo participación de peatón (y dado que hubo motocicleta)?
7. ¿Cuál es la probabilidad de que un accidente tenga heridos, dado que ocurrió en un barrio seleccionado?

**Ruta del laboratorio:** cargar la base → revisar columnas → identificar las variables de interés → limpiar lo mínimo necesario → definir eventos → calcular probabilidades → aplicar Bayes → interpretar.

# 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": .3})
AZUL, CORAL = "#00ADC6", "#FF6F49"

In [ ]:
# Funciones de apoyo para la presentación y para registrar respuestas
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

respuestas = {}

def bloque(titulo, texto, color="#00838f", fondo="#e6f7f9"):
    display(HTML(f"""
    <div style="background:{fondo};color:#1f2937;padding:14px 18px;border-radius:12px;border-left:6px solid {color};
                font-family:Arial, sans-serif;margin:10px 0;">
      <h3 style="color:{color};margin:0 0 6px 0;">{titulo}</h3>
      <p style="font-size:15px;line-height:1.55;margin:0;text-align:justify;">{texto}</p>
    </div>"""))

def pregunta(id_pregunta, enunciado):
    """Muestra una pregunta de análisis con una caja de texto y un botón para guardar la respuesta."""
    caja = widgets.Textarea(value=respuestas.get(id_pregunta, ""), placeholder="Escriba aquí su análisis...",
                            layout=widgets.Layout(width="100%", height="130px"))
    boton = widgets.Button(description="Guardar respuesta", button_style="success", icon="check")
    salida = widgets.Output()
    def guardar(_):
        respuestas[id_pregunta] = caja.value
        with salida:
            clear_output(); print("Respuesta guardada.")
    boton.on_click(guardar)
    display(HTML(f"""
    <div style="background:#fff4f0;color:#1f2937;padding:12px 16px;border-radius:10px;border:1px solid #ffc9b8;
                font-family:Arial, sans-serif;margin:10px 0;">
      <b style="color:#d9480f;">Pregunta de análisis {id_pregunta}</b>
      <p style="font-size:15px;line-height:1.5;margin:6px 0 0 0;">{enunciado}</p>
    </div>"""))
    display(caja, boton, salida)

# 2. Cargar la base de datos

La opción `descargar` trae la base directamente del portal. Si falla, descárguela en formato **CSV** desde el [portal](https://www.datos.gov.co/Transporte/3-Accidentes-de-Transito-ocurridos-en-el-Municipio/7cci-nqqb) (*Exportar → CSV*) y use `subir`.

In [ ]:
#@markdown ### **¿Cómo quiere cargar la base?**
#@markdown - `descargar`: la descarga directamente desde datos.gov.co (necesita internet; puede tardar).
#@markdown - `subir`: abre una ventana para subir el CSV que ya descargó (solo en Colab).
#@markdown - `archivo`: lee un CSV que ya está en la carpeta del notebook.
metodo = "descargar"  #@param ["descargar", "subir", "archivo"]
nombre_archivo = ""  #@param {type:"string"}
#@markdown ---
URL = "https://www.datos.gov.co/api/views/7cci-nqqb/rows.csv?accessType=DOWNLOAD"

if metodo == "descargar":
    print("Descargando Accidentes de tránsito de Bucaramanga desde datos.gov.co ...")
    df = pd.read_csv(URL, low_memory=False)
elif metodo == "subir":
    from google.colab import files
    subido = files.upload()
    df = pd.read_csv(next(iter(subido)), low_memory=False)
else:
    df = pd.read_csv(nombre_archivo, sep=None, engine="python")

print("Registros:", len(df), "| Variables:", df.shape[1])
df.head()

# 3. Explorar la base

Revise cuántos registros y variables tiene la base, cómo se llaman exactamente las columnas y qué valores toman. El teorema de Bayes exige definir los eventos a partir de **categorías reales** de los datos, así que este paso no se puede saltar.

In [ ]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
for c in df.columns:
    print(" -", c)

In [ ]:
#@markdown Muestra los valores más frecuentes de cada columna (puede ser largo)
for col in df.columns:
    print(f"\n=== {col} ===")
    print(df[col].astype(str).value_counts(dropna=False).head(8).to_string())

# 4. Identificar y limpiar las columnas de interés

Se normalizan los nombres de las columnas (mayúsculas, sin tildes) y se buscan las cinco variables del análisis. Después se estandariza el texto de las categorías para que, por ejemplo, `Con heridos`, `con heridos` y `CON HERIDOS ` queden iguales.

In [ ]:
def normalizar(texto):
    """Mayúsculas, sin tildes (conserva la Ñ) y sin espacios repetidos."""
    texto = " ".join(str(texto).upper().split())
    return texto.translate(str.maketrans("ÁÉÍÓÚÜÀÈÌÒÙ", "AEIOUUAEIOU"))

df.columns = [normalizar(c) for c in df.columns]

def buscar_columna(*opciones):
    for o in opciones:
        if o in df.columns:
            return o
    raise KeyError(f"No encontré ninguna de {opciones}. Revise la lista de columnas y escriba el nombre correcto.")

col_gravedad = buscar_columna("GRAVEDAD")
col_tiempo   = buscar_columna("DIURNIO/NOCTURNO", "DIURNO/NOCTURNO", "JORNADA")
col_moto     = buscar_columna("MOTO", "MOTOCICLETA")
col_peaton   = buscar_columna("PEATON")
col_barrio   = buscar_columna("BARRIO")
print("Columnas usadas:", col_gravedad, "|", col_tiempo, "|", col_moto, "|", col_peaton, "|", col_barrio)

In [ ]:
for c in [col_gravedad, col_tiempo, col_barrio]:
    df[c] = df[c].map(normalizar)

def participa(serie):
    """True si el actor vial participó: número > 0, o texto como SI / X / 1."""
    num = pd.to_numeric(serie, errors="coerce")
    texto = serie.astype(str).map(normalizar)
    return (num > 0) | texto.isin(["SI", "S", "X", "1"])

df_bayes = df[[col_gravedad, col_tiempo, col_barrio]].copy()
df_bayes["MOTO_PARTICIPA"] = participa(df[col_moto])
df_bayes["PEATON_PARTICIPA"] = participa(df[col_peaton])
df_bayes = df_bayes[(df_bayes[col_gravedad] != "NAN") & (df_bayes[col_tiempo] != "NAN")]
print("Registros utilizados:", len(df_bayes))
display(df_bayes[col_gravedad].value_counts().to_frame("frecuencia"))
display(df_bayes[col_tiempo].value_counts().to_frame("frecuencia"))

# 5. Ejemplo guiado: P(heridos | nocturno) con el teorema de Bayes

Se definen los eventos:

- **A:** el accidente tiene heridos (`GRAVEDAD == "CON HERIDOS"`)
- **B:** el accidente ocurrió en horario nocturno (`DIURNIO/NOCTURNO == "NOCTURNO"`)

Cada evento es una columna de valores `True`/`False`. Como Python cuenta `True` como 1, **la media de un evento es su frecuencia relativa**, es decir, la estimación de su probabilidad.

In [ ]:
valor_heridos, valor_nocturno = "CON HERIDOS", "NOCTURNO"
A = df_bayes[col_gravedad] == valor_heridos
B = df_bayes[col_tiempo] == valor_nocturno

P_A = A.mean()            # P(A)
P_B = B.mean()            # P(B)
P_B_dado_A = B[A].mean()  # P(B|A): entre los accidentes con heridos, proporción nocturna
print(f"P(A)   = {P_A:.4f}")
print(f"P(B)   = {P_B:.4f}")
print(f"P(B|A) = {P_B_dado_A:.4f}")

**Teorema de Bayes**

$$P(A\mid B) = \frac{P(B\mid A)\,P(A)}{P(B)}$$

Se calcula con la fórmula y se **verifica** con el cálculo directo: entre los accidentes nocturnos, ¿qué proporción tuvo heridos? Si los dos valores coinciden, la aplicación es correcta.

In [ ]:
P_A_dado_B = P_B_dado_A * P_A / P_B
P_A_dado_B_directa = A[B].mean()
print(f"P(A|B) por Bayes   = {P_A_dado_B:.4f}  ({P_A_dado_B:.2%})")
print(f"P(A|B) directa     = {P_A_dado_B_directa:.4f}  ({P_A_dado_B_directa:.2%})")

In [ ]:
tabla = pd.crosstab(df_bayes[col_gravedad], df_bayes[col_tiempo], margins=True, margins_name="Total")
print("Frecuencias conjuntas")
display(tabla)
print("Porcentaje por columna (gravedad dentro de cada horario)")
display((100 * pd.crosstab(df_bayes[col_gravedad], df_bayes[col_tiempo], normalize="columns")).round(2))

### Simulador interactivo

In [ ]:
def barra(valor):
    pct = max(0, min(valor * 100, 100))
    color = "#00ADC6" if pct < 33 else "#F2B134" if pct < 66 else "#FF6F49"
    return f"""<div style="width:100%;background:#e9ecef;border-radius:10px;overflow:hidden;height:28px;margin-top:8px">
      <div style="width:{pct}%;background:{color};height:28px;text-align:center;font-weight:bold;line-height:28px;color:#1f2937">{pct:.1f}%</div></div>"""

def simulador_bayes(P_A, P_B, P_B_dado_A):
    if P_B == 0:
        display(HTML("<b>P(B) no puede ser 0.</b>")); return
    post = P_B_dado_A * P_A / P_B
    if post > 1:
        nota = "<div style='background:#fff3cd;color:#1f2937;padding:10px;border-radius:8px'>Los valores no son coherentes: P(A|B) superaría 1. Recuerde que P(B) ≥ P(B|A)·P(A).</div>"
        post_mostrar = 1
    else:
        nivel = "baja" if post < .33 else "moderada" if post < .66 else "alta"
        cambio = "aumenta" if post > P_A else "disminuye" if post < P_A else "no cambia"
        nota = f"<div style='background:#e6f7f9;color:#1f2937;padding:10px;border-radius:8px'><b>Interpretación:</b> la probabilidad posterior es {nivel}. Al saber que ocurrió B, la probabilidad de A {cambio} respecto a la previa ({P_A:.1%}).</div>"
        post_mostrar = post
    display(HTML(f"""
    <div style="font-family:Arial;background:#f8f9fa;color:#1f2937;padding:16px;border-radius:14px;border:1px solid #dee2e6;max-width:820px">
      <h3 style="color:#00838f;margin:0 0 6px 0">Simulador del teorema de Bayes</h3>
      <p style="margin:0"><b>A:</b> accidente con heridos &nbsp;·&nbsp; <b>B:</b> accidente en horario nocturno</p>
      <p style="font-size:17px;margin:10px 0">P(A|B) = P(B|A)·P(A) / P(B) = ({P_B_dado_A:.4f} × {P_A:.4f}) / {P_B:.4f} = <b style="color:#d9480f">{post:.4f}</b></p>
      {barra(post_mostrar)}<div style="margin-top:10px">{nota}</div>
    </div>"""))

s_A = widgets.FloatSlider(value=float(P_A), min=.01, max=1, step=.01, description="P(A)")
s_B = widgets.FloatSlider(value=float(P_B), min=.01, max=1, step=.01, description="P(B)")
s_BA = widgets.FloatSlider(value=float(P_B_dado_A), min=.01, max=1, step=.01, description="P(B|A)")
display(widgets.VBox([widgets.HTML("<b>Mueva los deslizadores para ver cómo cambia la probabilidad posterior</b>"), s_A, s_B, s_BA]),
        widgets.interactive_output(simulador_bayes, {"P_A": s_A, "P_B": s_B, "P_B_dado_A": s_BA}))

# 6. Actividad: calcule las probabilidades del caso

Defina los eventos y complete las celdas. Use la media de los eventos lógicos para las probabilidades simples y el filtrado `X[Y].mean()` para las condicionales P(X | Y).

| Evento | Definición |
|---|---|
| A | accidente con heridos |
| B | accidente en horario nocturno |
| C | participación de motocicleta |
| D | participación de peatón |
| E | accidente con muertos |
| F | accidente en el barrio seleccionado |

In [ ]:
print("Barrios con más accidentes:")
print(df_bayes[col_barrio].value_counts().head(15).to_string())

In [ ]:
#@markdown ### **Elija el barrio que va a analizar** (escríbalo como aparece en la lista)
barrio_seleccionado = ""  #@param {type:"string"}
#@markdown ---
if not barrio_seleccionado.strip():
    barrio_seleccionado = df_bayes[col_barrio].value_counts().index[0]
barrio_seleccionado = normalizar(barrio_seleccionado)
print("Barrio seleccionado:", barrio_seleccionado, "| accidentes:", (df_bayes[col_barrio] == barrio_seleccionado).sum())

In [ ]:
# Defina los eventos C, D, E y F como series de True/False
C = ...   # participación de motocicleta  (use la columna "MOTO_PARTICIPA")
D = ...   # participación de peatón       (use la columna "PEATON_PARTICIPA")
E = ...   # accidente con muertos
F = ...   # accidente en el barrio seleccionado

In [ ]:
# Calcule cada probabilidad y guárdela en el diccionario
resultados = {
    "1. P(A) heridos": ...,
    "2. P(B) nocturno": ...,
    "3. P(A|B) heridos dado nocturno": ...,
    "4. P(B|A) nocturno dado heridos": ...,
    "5. P(A|C) heridos dado motocicleta": ...,
    "6a. P(E|D) muertos dado peatón": ...,
    "6b. P(E|C) muertos dado motocicleta": ...,
    "7. P(A|F) heridos dado barrio": ...,
    "Referencia: P(E) muertos": ...,
}
tabla_resultados = pd.DataFrame({"probabilidad": resultados})
tabla_resultados["porcentaje"] = (100 * tabla_resultados["probabilidad"]).round(2)
tabla_resultados

In [ ]:
# Gráfico de apoyo: probabilidad de heridos según cada condición
comparacion = pd.Series({"Todos": A.mean(), "Nocturno": A[B].mean(), "Diurno": A[~B].mean(),
                         "Con moto": A[C].mean(), "Sin moto": A[~C].mean(),
                         "Con peatón": A[D].mean(), f"Barrio\n{barrio_seleccionado[:14]}": A[F].mean()})
ax = comparacion.plot(kind="bar", color=[CORAL] + [AZUL] * (len(comparacion) - 1))
ax.axhline(A.mean(), color=CORAL, ls="--", lw=1)
ax.set_ylabel("P(heridos | condición)"); ax.set_title("¿En qué condiciones es más probable un accidente con heridos?")
plt.xticks(rotation=0); plt.show()

# 7. Preguntas de análisis

In [ ]:
pregunta("1", "Compare la probabilidad general de accidentes con heridos, P(A), con la probabilidad de heridos en horario nocturno, P(A|B). ¿Qué puede concluir sobre la influencia del horario en la gravedad del accidente?")

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

In [ ]:
pregunta("2", "Compare P(A|B), heridos dado nocturno, con P(B|A), nocturno dado heridos. ¿Cuál es mayor? ¿Qué indica esta diferencia sobre la relación entre el horario y la gravedad?")

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

In [ ]:
pregunta("3", "Analice la probabilidad de accidentes con muertos cuando hay participación de motocicleta, P(E|C), y compárela con P(E) y con P(E|D) (peatón). ¿Qué indica este resultado sobre el riesgo asociado a estos actores viales?")

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

In [ ]:
pregunta("4", "Compare la probabilidad de accidentes con heridos en el barrio seleccionado, P(A|F), con la probabilidad general P(A). ¿Este barrio presenta mayor incidencia de accidentes con heridos?")

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

In [ ]:
pregunta("5", "A partir de los resultados, ¿cuál de las variables analizadas (horario, tipo de actor vial o ubicación) parece tener mayor relación con la gravedad de los accidentes? Justifique con las probabilidades calculadas.")

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

In [ ]:
#@markdown # 8. Descargar las respuestas
TITULO_ENTREGA = "Respuestas · Laboratorio 2 · Teorema de Bayes"
ARCHIVO_ENTREGA = "Respuestas_Laboratorio_2.docx"

# Genera un documento Word con las respuestas guardadas y lo descarga
try:
    from docx import Document
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"], check=True)
    from docx import Document

documento = Document()
documento.add_heading(TITULO_ENTREGA, level=1)
if not respuestas:
    documento.add_paragraph("No se registraron respuestas.")
for id_preg, texto in respuestas.items():
    documento.add_heading(f"Pregunta {id_preg}", level=2)
    documento.add_paragraph(texto.strip() or "Sin respuesta registrada.")
documento.save(ARCHIVO_ENTREGA)
print("Archivo creado:", ARCHIVO_ENTREGA)
try:
    from google.colab import files
    files.download(ARCHIVO_ENTREGA)
except ImportError:
    print("Fuera de Colab: el archivo quedó en la carpeta del notebook.")